## 1. 安裝必要套件與 WanGP


In [ ]:
'''
Install packages required for the notebook interface and INT8 offload.
安裝 Notebook 互動介面、INT8 與動態 Offload 所需套件。
'''
%pip -q install "ipywidgets>=8,<9" "mmgp==3.8.2" "comfy-kitchen==0.2.35" "transformers==4.54.0" "diffusers==0.36.0" "gradio==5.50.0" "ffmpeg-python" "optimum-quanto" "accelerate" "einops" "hf_xet" "safetensors" "psutil"

'''
Download the pinned WanGP source only when it does not already exist.
僅在程式碼尚未存在時下載固定版本 WanGP。
'''
!test -d /kaggle/working/WanGP/.git || git clone -q https://github.com/deepbeepmeep/WanGP.git /kaggle/working/WanGP

'''
Keep the API-compatible version; newer WanGP versions may change private Python APIs. Fetch the implementation matched to the model-loading functions below.
保留相容的固定版本；新版 WanGP 可能變更內部 Python API。 下載與以下模型載入程式相容的固定提交版本。
'''
!git -C /kaggle/working/WanGP fetch -q --depth 1 origin 91301f0eda923b6c4b05159e32f1dfd2a960af9d

'''
Keep the source stable across repeated notebook runs.
固定原始碼版本，避免反覆執行時發生不預期變動。
'''
!git -C /kaggle/working/WanGP checkout -q --detach FETCH_HEAD

'''
Record the commit used by the hardcoded WanGP Qwen-2.1 pipeline.
記錄目前 Notebook 固定的 WanGP Qwen-2.1 Pipeline 版本。
'''
!git -C /kaggle/working/WanGP rev-parse --short HEAD


## 2. 模型精度與 GPU 配置


In [ ]:
'''
Select the checkpoint precision and hardware arrangement. Available choices: int8_single, bf16_single, bf16_dual.
選擇權重精度與 GPU 分工模式。 可選值：int8_single、bf16_single、bf16_dual。
'''
MODEL_MODE = "bf16_dual"

'''
Use GPU 0 for image generation.
使用 GPU 0 執行圖片生成。
'''
GPU_ID = 0

'''
Set GPU 0 Transformer weight preload budget in GiB.
設定 GPU 0 Transformer 權重預載預算（GiB）。
'''
GPU0_MODEL_GIB = 14.0

'''
Reject unsupported model modes before large downloads.
下載大型模型之前先確認模式有效。
'''
assert MODEL_MODE in ("int8_single", "bf16_single", "bf16_dual")

'''
Display the selected configuration.
顯示目前選用的配置。
'''
print(f"Selected mode: {MODEL_MODE}")


## 3. GPU 0 Transformer 預載預算


In [ ]:
'''
Ensure Cell 2 has supplied the GPU 0 memory budget.
確認 Cell 2 已設定 GPU 0 的記憶體預算。
'''
assert "GPU0_MODEL_GIB" in globals(), "請先執行 Cell 2，設定 GPU0_MODEL_GIB。"

'''
Import memory and validation helpers.
載入記憶體管理與數值檢查工具。
'''
import gc
import math
import torch

'''
Reject malformed values but leave the budget user-controlled.
只檢查數值有效性，不限制指定的容量。
'''
assert isinstance(GPU0_MODEL_GIB, (int, float)) and math.isfinite(GPU0_MODEL_GIB) and GPU0_MODEL_GIB > 0

'''
Check whether the existing model can be reprofiled in this session.
確認目前 Session 是否已經載入可重新配置的模型。
'''
state = globals().get("QWEN21_STATE")

'''
Keep a newly chosen budget for the first model initialization.
若模型尚未初始化，先把設定值記住，供後續載入使用。
'''
globals()["GPU0_MODEL_GIB"] = float(GPU0_MODEL_GIB)

'''
Stop here when no model has been initialized yet.
如果模型尚未初始化，先顯示目前預算即可。
'''
if not state:
    print(f"GPU 0 Transformer 預載預算已設定為 {GPU0_MODEL_GIB:.1f} GiB。")
    print("初始化模型後，若你再修改這個值，請重新執行本格即可套用。")
else:

    '''
    Ensure the state has the minimum fields required for live reprofiling.
    確認狀態內含有重新配置所需的欄位。
    '''
    required = ("managed", "transformer", "dtype", "lora_path")
    assert all(key in state for key in required), "目前模型狀態不支援即時重配置，請重新初始化模型。"

    previous_gib = float(state.get("gpu0_model_gib", GPU0_MODEL_GIB))

    '''
    Prevent generation using a manager that has already been released.
    避免生成 Cell 使用已釋放的管理器。
    '''
    state["manager"] = None

    '''
    Clear unused allocations before creating the new MMGP memory profile.
    建立新預算前清理未使用的 CUDA 配置。
    '''
    gc.collect()
    torch.cuda.empty_cache()

    '''
    Reprofile existing model weights without downloading checkpoints or restarting.
    利用既有模型權重重新配置，不重新下載、不重啟 Kernel。
    '''
    try:
        updated = configure_gpu0_manager(
            state["managed"], state["transformer"],
            round(GPU0_MODEL_GIB * 1024), state["dtype"], state["lora_path"],
        )
    except Exception as error:
        try:
            state["manager"] = configure_gpu0_manager(
                state["managed"], state["transformer"],
                round(previous_gib * 1024), state["dtype"], state["lora_path"],
            )
        except Exception as recovery_error:
            raise RuntimeError("MMGP 更新及復原均失敗，請檢查 RAM 與模型狀態。") from recovery_error
        raise RuntimeError(f"套用失敗；已恢復 {previous_gib:.1f} GiB。") from error
    else:
        state["manager"] = updated
        state["gpu0_model_gib"] = float(GPU0_MODEL_GIB)
        manager = updated
        print(f"GPU 0 Transformer 預載預算已更新：{previous_gib:.1f} → {GPU0_MODEL_GIB:.1f} GiB")

    free_bytes, total_bytes = torch.cuda.mem_get_info(GPU_ID)
    print(f"GPU 0：模型預算 {state['gpu0_model_gib']:.1f} GiB；目前總占用 {(total_bytes-free_bytes)/1024**3:.2f} GiB")
    print("預算不是保證占用量；請在下一次生成時確認 VRAM 峰值。")


## 4. 依所選模式下載模型


In [ ]:
'''
Import filesystem helpers before configuring the Hugging Face cache.
在設定 Hugging Face 快取之前，先載入檔案系統工具。
'''
import os
from pathlib import Path
import shutil

'''
Store Hugging Face's cache in Kaggle's temporary storage.
將 Hugging Face 快取集中在 Kaggle 的暫存磁碟。
'''
os.environ["HF_HOME"] = "/tmp/qwen21_hf"
os.environ["HF_HUB_CACHE"] = "/tmp/qwen21_hf/hub"

'''
Disable the optional Xet chunk cache to reduce duplicated downloads.
關閉非必要的 Xet 分塊快取，減少重複占用磁碟。
'''
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

'''
Import the downloader after the cache location has been configured.
設定好快取路徑後，再載入模型下載工具。
'''
from huggingface_hub import hf_hub_download

'''
Save model checkpoints under /tmp rather than /kaggle/working.
將模型權重存放於 /tmp，不占用 /kaggle/working 空間。
'''
CKPT = Path("/tmp/qwen21_ckpts")
CKPT.mkdir(parents=True, exist_ok=True)

'''
Show temporary-disk capacity before downloading large weights.
下載大型權重之前，顯示暫存磁碟的可用空間。
'''
free_gib = shutil.disk_usage(CKPT).free / 1024**3
print(f"Models: {CKPT} | Available disk space: {free_gib:.1f} GiB")

'''
Choose the correctly packaged WanGP transformer checkpoint.
選用 WanGP 專用格式的 Transformer 權重，避免與 ComfyUI 權重混用。
'''
TRANSFORMER_FILE = (
    "qwen_image_21_7B_int8_convrot.safetensors"
    if MODEL_MODE == "int8_single"
    else "qwen_image_21_7B_bf16.safetensors"
)

'''
Download the selected transformer, quantized encoder, VAE, and optional accelerator.
下載指定 Transformer、INT8 編碼器、VAE 與加速用 LoRA。
'''
MODEL_FILES = [
    ("DeepBeepMeep/Qwen_image_2", TRANSFORMER_FILE),
    ("DeepBeepMeep/Ideogram4", "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors"),
    ("DeepBeepMeep/Qwen_image_2", "qwen_image_21/qwen_image_21_vae.safetensors"),
    ("DeepBeepMeep/Qwen_image_2", "qwen_image_21/vae_config.json"),
    ("DeepBeepMeep/Qwen_image_2", "qwen_image_21/scheduler_config.json"),
    ("Viggle/Qwen-Image-2.1-viggle-turbo", "Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors"),
]

'''
Fetch the original processor/tokenizer files required by the model loader.
下載模型載入器需要的 tokenizer 與影像處理設定檔。
'''
ENCODER_CONFIGS = [
    "config_legacy.json", "tokenizer_legacy.json", "tokenizer_config_legacy.json",
    "preprocessor_config.json", "chat_template.jinja", "merges.txt", "vocab.json",
    "added_tokens.json", "special_tokens_map.json", "video_preprocessor_config.json",
]
MODEL_FILES.extend(("DeepBeepMeep/Ideogram4", f"Qwen3-VL-8B-Instruct/{name}") for name in ENCODER_CONFIGS)

'''
Reuse previously downloaded files within the same Kaggle runtime.
同一個 Kaggle 執行環境內會沿用已下載檔案，避免重複下載。
'''
for repo, filename in MODEL_FILES:
    print(f"Checking: {filename}", flush=True)
    hf_hub_download(repo_id=repo, filename=filename, local_dir=CKPT, cache_dir=os.environ["HF_HUB_CACHE"])

'''
Read the encoder's checkpoint metadata using the same helper as WanGP.
使用 WanGP 相同的工具檢查編碼器權重是否已包含視覺模型。
'''
from mmgp import quant_router
encoder_path = CKPT / "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors"
encoder_keys, _ = quant_router.load_metadata_state_dict(str(encoder_path))

'''
Download the separate vision encoder only if the INT8 checkpoint omits it.
若 INT8 權重未包含視覺編碼器，才下載獨立的 BF16 視覺權重。
'''
if not any(key.startswith(("visual.", "model.visual.")) for key in encoder_keys):
    vision_file = "Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_vision_bf16.safetensors"
    print(f"Checking separate vision weights: {vision_file}", flush=True)
    hf_hub_download(repo_id="DeepBeepMeep/Ideogram4", filename=vision_file, local_dir=CKPT, cache_dir=os.environ["HF_HUB_CACHE"])

del encoder_keys

'''
Confirm the mode-specific model is present before expensive initialization.
執行耗時的初始化之前先確認所需權重已準備好。
'''
assert (CKPT / TRANSFORMER_FILE).is_file(), f"Missing: {TRANSFORMER_FILE}"
print(f"Ready: {TRANSFORMER_FILE}")


## 5. 模型初始化


In [ ]:
'''
Import standard utilities and PyTorch.
載入標準工具與 PyTorch。
'''
import os
import sys
import types
import torch

'''
Disable unrelated frameworks and tokenizer thread oversubscription.
關閉不需要的框架並避免 tokenizer 過度使用執行緒。
'''
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

'''
Check CUDA availability and the number of devices before touching models.
在載入模型之前確認 GPU 數量足夠。
'''
assert torch.cuda.is_available(), "請在 Kaggle Notebook Settings 開啟 GPU。"
assert torch.cuda.device_count() > GPU_ID, f"找不到 GPU {GPU_ID}。"
assert MODEL_MODE != "bf16_dual" or torch.cuda.device_count() >= 2, "bf16_dual 需要兩張 GPU。"

'''
Show a RAM warning before loading the large BF16 checkpoint.
載入大型 BF16 權重前，先顯示系統記憶體不足的風險。
'''
if MODEL_MODE != "int8_single":
    import psutil
    available_ram_gib = psutil.virtual_memory().available / 1024**3
    print(f"Available host RAM: {available_ram_gib:.1f} GiB")
    if available_ram_gib < 25:
        print("Warning: BF16 transformer + INT8 text encoder can exhaust Kaggle RAM. Consider int8_single.")

'''
T4 uses FP16 compute; it cannot execute BF16 operations natively.
T4 不支援原生 BF16 運算，因此使用 FP16 執行運算。
'''
DEVICE = f"cuda:{GPU_ID}"
EXEC_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(GPU_ID)[0] >= 8 else torch.float16

'''
Prevent using a different model mode with an already loaded pipeline.
模型載入後不可在同一個 Kernel 直接切換精度，以免重複權重占滿 RAM。
'''
existing = globals().get("QWEN21_STATE")
if existing and existing.get("pipeline") is not None:
    assert existing["mode"] == MODEL_MODE, "切換 MODEL_MODE 後請重新啟動 Kernel，再執行下載與初始化 Cell。"

    '''
    A released MMGP manager cannot be safely reused.
    已釋放的 MMGP 管理器不能直接重複使用。
    '''
    assert existing.get("manager") is not None, "MMGP 管理器已失效，不能直接沿用舊的 Pipeline。"
    pipeline = existing["pipeline"]
    manager = existing["manager"]
    parts = existing["parts"]
    print(f"Reusing {MODEL_MODE} model without reloading weights.")
else:

    '''
    Keep MMGP execution on GPU 0, except the explicitly separated text encoder.
    MMGP 預設管理 GPU 0；雙 GPU 模式的文字編碼器另行放在 GPU 1。
    '''
    torch.cuda.set_device(GPU_ID)

    '''
    Import the already installed source directly, without writing a worker script.
    直接匯入已安裝的程式碼，不額外產生 .py Worker。
    '''
    sys.path.insert(0, "/kaggle/working/WanGP")
    os.chdir("/kaggle/working/WanGP")

    '''
    Stub the optional rembg dependency because background removal is not used.
    未使用背景移除功能，因此建立輕量的 rembg 相容模組。
    '''
    if "rembg" not in sys.modules:
        rembg = types.ModuleType("rembg")
        def unavailable_rembg(*args, **kwargs):
            raise RuntimeError("本 Notebook 未啟用 rembg。")
        rembg.remove = rembg.new_session = unavailable_rembg
        sys.modules["rembg"] = rembg

    '''
    Load WanGP's exact quantization handler and Qwen 2.1 pipeline.
    載入 WanGP 對應的量化處理器及 Qwen 2.1 Pipeline。
    '''
    from mmgp import offload, quant_router
    from optimum.quanto.tensor.weights import qbytes
    from shared.utils import files_locator
    from shared.kernels import int8_backend, quanto_int8_inject
    from models.qwen21.qwen21_handler import family_handler, ENCODER

    '''
    Activate the INT8 ConvRot format and local model locator.
    啟用 INT8 ConvRot 格式與模型搜尋路徑。
    '''
    quant_router.unregister_handler(".fp8_quanto_bridge")
    quant_router.register_handler("shared.qtypes.int8_convrot")
    files_locator.set_checkpoints_paths([str(CKPT)])

    '''
    Configure the INT8 kernel backend for the selected hardware.
    依硬體設定 INT8 運算後端。
    '''
    int8_backend.configure("auto", 1)
    int8_backend._SCRATCH_BYTES = 384 * 1024 * 1024

    '''
    Keep the large token embedding table on the CPU.
    將大型 token embedding 表保留在 CPU，降低 GPU 記憶體用量。
    '''
    class CPUEmbedding(torch.nn.Module):
        def __init__(self, weight):
            super().__init__()
            self.table = [weight]
        def forward(self, input_ids):
            return torch.nn.functional.embedding(input_ids.cpu(), self.table[0]).to(input_ids.device)

    '''
    Preserve BF16 original weights in host RAM; let MMGP cast their compute copy.
    BF16 原始權重先保留於 RAM，實際運算副本交由 MMGP 轉成 FP16。
    '''
    original_loader = offload.fast_load_transformers_model
    def load_with_dtype(*args, **kwargs):
        requested = args[0] if args else kwargs.get("model_path", "")
        is_bf16_transformer = str(requested).endswith("_7B_bf16.safetensors")
        kwargs.setdefault("default_dtype", torch.bfloat16 if is_bf16_transformer else EXEC_DTYPE)
        return original_loader(*args, **kwargs)
    offload.fast_load_transformers_model = load_with_dtype

    '''
    Load the matching checkpoint and Qwen3-VL encoder once.
    使用相容權重與 Qwen3-VL 編碼器進行單次初始化。
    '''
    try:
        model_path = CKPT / TRANSFORMER_FILE
        assert model_path.exists(), f"缺少權重：{model_path}"
        model_def = family_handler.query_model_def("qwen_image_21_7B", {})
        pipeline, parts = family_handler.load_model(
            [str(model_path)], "qwen_image_21_7B", "qwen_image_21_7B", model_def,
            text_encoder_filename=str(CKPT / ENCODER / f"{ENCODER}_int8_convrot.safetensors"),
        )
    finally:
        offload.fast_load_transformers_model = original_loader

    '''
    Separate the actual model submodules for device placement.
    取出各個模型元件，決定它們使用哪張 GPU。
    '''
    transformer = parts["transformer"]
    text_encoder = parts["text_encoder"]
    vision_encoder = parts["vision_encoder"]
    vae = parts["vae"]
    text_encoder.embed_tokens = CPUEmbedding(text_encoder.embed_tokens.weight.data)

    '''
    Convert the BF16 transformer compute copy to FP16 on T4.
    T4 運算時會將 BF16 Transformer 副本轉為 FP16。
    '''
    transformer._convertWeightsFloatTo = EXEC_DTYPE
    text_encoder._convertWeightsFloatTo = EXEC_DTYPE
    vision_encoder._convertWeightsFloatTo = EXEC_DTYPE

    '''
    Keep VAE in FP32 to avoid FP16 numerical overflow.
    保留 VAE 的 FP32 精度，避免 FP16 數值溢位。
    '''

    vae._convertWeightsFloatTo = None

    '''
    In dual mode, keep the quantized text/vision encoder on GPU 1.
    雙 GPU 模式將 INT8 文字／視覺編碼器放在 GPU 1，GPU 0 保留給生成。
    '''
    if MODEL_MODE == "bf16_dual":
        pipeline.text_encoder.to("cuda:1")
        print("GPU 1: Qwen3-VL INT8 encoder placed on cuda:1.")
        managed = {"transformer": transformer, "vae": vae}
    else:
        managed = {
            "transformer": transformer, "text_encoder": text_encoder,
            "vision_encoder": vision_encoder, "vae": vae,
        }

    '''
    Configure MMGP from existing model objects; this is reused for live budget changes.
    使用既有模型物件建立 MMGP；後續變更預算時可重複使用，不重新讀取主模型權重。
    '''
    def configure_gpu0_manager(modules, generator, budget_mb, dtype, lora_file):

        '''
        Allocate the selected approximate VRAM budget for the generator.
        配置生成模型的預估 VRAM 預算，並保留其它運算空間。
        '''
        configured = offload.profile(
            modules,
            profile_no=5,
            pinnedMemory=False,
            quantizeTransformer=False,
            extraModelsToQuantize=[],
            budgets={"transformer": budget_mb, "vae": 2048, "text_encoder": 2048, "vision_encoder": 1536},

            vram_safety_coefficient=0.95,
            loras=["transformer"],
            convertWeightsFloatTo=dtype,
            asyncTransfers=True,
        )

        '''
        Reapply the Turbo LoRA after a manager has been configured.
        每次 MMGP 設定完成後重新掛載 Turbo LoRA，避免設定更新後遺失。
        '''
        try:

            '''
            Load the small adapter without reloading the main transformer checkpoint.
            只重新讀取小型 adapter，不重新載入 Transformer 主權重檔。
            '''
            loaded = offload.load_loras_into_model(
                generator, [str(lora_file)], [1.0], activate_all_loras=True,
                pinnedLora=False,
                preprocess_sd=lambda sd: generator.preprocess_loras("qwen_image_21_7B", sd),
                split_linear_modules_map=getattr(generator, "split_linear_modules_map", None),
            )

            '''
            Reject a partial configuration without the required adapter.
            若 Turbo LoRA 未成功掛載，不使用不完整的模型設定。
            '''
            assert loaded, "Turbo LoRA 載入失敗。"

            '''
            Enable the same adapter weights used by the 6-step sampler.
            啟用六步採樣器搭配的 LoRA 權重。
            '''
            offload.activate_loras(generator, ["0"], [1.0])
        except Exception:

            '''
            Release an incomplete manager before retrying or reporting failure.
            若設定未完成，釋放 MMGP 管理器，避免重試時殘留資源。
            '''
            configured.release()
            raise

        '''
        Return the configured manager while retaining the original model objects.
        回傳完成設定的管理器，原本的模型物件仍維持不變。
        '''
        return configured

    '''
    Locate the already downloaded six-step Turbo adapter.
    找出已下載的六步 Turbo adapter。
    '''
    lora_path = CKPT / "Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors"

    '''
    Build the manager once, without creating another complete model copy.
    首次建立管理器，不額外建立第二份完整模型。
    '''
    manager = configure_gpu0_manager(
        managed, transformer, round(GPU0_MODEL_GIB * 1024),
        EXEC_DTYPE, lora_path,
    )

    '''
    Preserve the same attention backend across later memory updates.
    之後更新記憶體設定時仍沿用相同 Attention backend。
    '''
    offload.shared_state["_attention"] = "sdpa"

    '''
    Run text encoding on its assigned GPU, then transfer conditioning to GPU 0.
    在指定 GPU 執行文字編碼，再將條件張量移回 GPU 0。
    '''
    original_encoder = pipeline._encode_conditioning
    def encode_prompt(prompts, images, device, batch_size):
        text_device = torch.device("cuda:1" if MODEL_MODE == "bf16_dual" else DEVICE)
        backend = int8_backend._backend
        original_forward = qbytes.WeightQBytesLinearFunction.__dict__["forward"]
        int8_backend._backend = "pytorch"
        qbytes.WeightQBytesLinearFunction.forward = staticmethod(
            quanto_int8_inject._default_quanto_qbytes_linear_forward
        )
        try:
            contexts = original_encoder(prompts, images, text_device, batch_size)
        finally:
            int8_backend._backend = backend
            qbytes.WeightQBytesLinearFunction.forward = original_forward
        converted = []
        for embeddings, mask, slots in contexts:
            if EXEC_DTYPE == torch.float16:
                peak = embeddings.float().abs().amax(dim=-1, keepdim=True)
                embeddings = embeddings.float() / (peak / 32768).clamp(min=1)
            converted.append((
                embeddings.to(device=DEVICE, dtype=EXEC_DTYPE),
                None if mask is None else mask.to(DEVICE),
                slots.to(DEVICE) if hasattr(slots, "to") else slots,
            ))
        return converted
    pipeline._encode_conditioning = encode_prompt

    '''
    Cache the initialized pipeline; later generations reuse the same objects.
    保留已建立的模型物件，後續生成不重新初始化。
    '''
    QWEN21_STATE = {
        "pipeline": pipeline, "manager": manager, "parts": parts,
        "mode": MODEL_MODE, "device": DEVICE, "dtype": EXEC_DTYPE,
        "managed": managed, "gpu0_model_gib": GPU0_MODEL_GIB,
        "lora_path": lora_path,
        "transformer": transformer,
    }

'''
Report the actual GPU configuration without claiming every weight is resident.
顯示實際使用的 GPU，避免將動態 Offload 誤稱為完整 GPU 常駐。
'''
print(f"Ready: {MODEL_MODE} | compute dtype={EXEC_DTYPE}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | allocated={torch.cuda.memory_allocated(i)/1024**3:.2f} GiB | free={free/1024**3:.2f} GiB")
print("Initialized model objects are kept. MMGP may dynamically move transformer blocks between RAM and VRAM.")


## 6. 參考圖片上傳與選取


In [ ]:
'''
Provide a multi-file upload control for the supported reference-image task.
為支援參考圖片的模式建立多選上傳介面。
'''
import ipywidgets as widgets
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO
from uuid import uuid4
from html import escape
from IPython.display import display, clear_output
import shutil

'''
Keep the selected image batch independent of other generation modes.
參考圖資料夾獨立保存，避免干擾其他生成模式。
'''
_REF_ROOT = Path('/kaggle/working/qwen21_references')
_REF_ROOT.mkdir(parents=True, exist_ok=True)
_REF_MAX = 10
_REF_FILES = sorted(p for p in _REF_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.webp'))
if len(_REF_FILES) > _REF_MAX or any(p.stem != f'{i:03d}' for i,p in enumerate(_REF_FILES,1)):
    raise RuntimeError('參考圖編號不連續；請清理舊資料夾再執行上傳 Cell。')

'''
Synchronize the numbered paths with the generation configuration.
將最新圖片清單同步到生成工作所使用的設定。
'''
def _ref_sync(paths):
    global REFERENCE_PATHS
    REFERENCE_PATHS = [str(path) for path in paths]
    pass

'''
Replace all reference images only after the new upload is fully validated.
新圖片全部通過驗證後才取代上一批；選得更少時也會清除多餘舊檔。
'''
def _ref_update(change):
    items = change['new']
    if not items:
        return
    if isinstance(items, dict):
        items = [dict(name=name, **item) for name, item in items.items()]
    stage = _REF_ROOT.with_name(_REF_ROOT.name + '_stage_' + uuid4().hex)
    stage.mkdir(parents=True)
    backup = _REF_ROOT.with_name(_REF_ROOT.name + '_backup_' + uuid4().hex)
    try:
        if not 1 <= len(items) <= _REF_MAX:
            raise ValueError(f'一次請選擇 1～{_REF_MAX} 張圖片。')
        for index, item in enumerate(items, 1):
            filename = Path(item['name']).name
            suffix = Path(filename).suffix.lower()
            if suffix not in ('.png','.jpg','.jpeg','.webp'):
                raise ValueError(f'不支援的圖片格式：{filename}')
            blob = bytes(item['content'])
            if len(blob) > 20 * 1024 * 1024:
                raise ValueError(f'單張圖片超過 20 MiB：{filename}')
            with Image.open(BytesIO(blob)) as photo:
                photo.verify()
            (stage / f'{index:03d}{suffix}').write_bytes(blob)
        _REF_ROOT.rename(backup)
        try:
            stage.rename(_REF_ROOT)
        except Exception:
            backup.rename(_REF_ROOT)
            raise
        shutil.rmtree(backup)
        current = sorted(_REF_ROOT.iterdir())
        _ref_sync(current)
        print('最新參考圖：', ', '.join(f'{p.name} = <Picture {i}>' for i,p in enumerate(current,1)))
    except Exception as error:
        shutil.rmtree(stage, ignore_errors=True)
        print('上傳未生效，保留上一批參考圖：', error)
    finally:
        _REF_PANEL.children = (_ref_new(),)

'''
Reset the browser picker so selecting the same filenames works again.
每次上傳後重建選檔元件，確保可以重新選相同檔名。
'''
def _ref_new():
    picker = widgets.FileUpload(accept='image/*', multiple=True, description='多選參考圖片')
    picker.observe(_ref_update, names='value')
    return picker

'''
Show a single picker for the complete current selection.
顯示目前整批參考圖的多選介面。
'''
_REF_PANEL = widgets.VBox()
_REF_PANEL.children = (_ref_new(),)
_ref_sync(_REF_FILES)
display(_REF_PANEL)
print('目前圖片：', ', '.join(p.name for p in _REF_FILES) or '尚未上傳')

## 7. 生成設定（T2I／Edit／Multi）


In [ ]:
'''
Choose the generation mode manually; selecting images never overrides this value. Available choices: text, edit, multi.
手動指定生成模式；選取參考圖不會更動此設定。 可選值：text、edit、multi。
'''
TASK = "text"

'''
Describe the image you want to generate or edit.
輸入要生成或編輯的圖片內容。
'''
PROMPT = '''
A cozy bookstore in Taipei on a rainy evening, warm cinematic lighting, realistic photography
'''.strip()

'''
Leave the negative prompt disabled by default.
預設不啟用負面提示詞。
'''
NEGATIVE_PROMPT = None

'''
Set the final output resolution.
設定最終輸出圖片的解析度。
'''
WIDTH = 640
HEIGHT = 640

'''
Set a reproducible seed, or -1 for a fresh seed on each run.
設定可重現種子；-1 代表每次生成都使用新種子。
'''
SEED = 42

'''
Set the reference-image preprocessing size.
設定參考圖片前處理解析度。
'''
REFERENCE_SIZE = 512

'''
Set the output filename.
設定輸出檔名。
'''
OUTPUT_NAME = "qwen21_output.png"

'''
Select the sampling strategy. Available choices: auto, turbo, base.
選擇採樣模式。 可選值：auto、turbo、base。
'''
SAMPLER = "auto"

'''
Set classifier-free guidance.
設定 CFG 引導強度。
'''
CFG = 1.0

'''
Set the sampling steps.
設定採樣步數。
'''
STEPS = 6


## 8. 生成圖片


In [ ]:
'''
Import runtime display and image conversion tools.
載入生成進度、圖片轉換與預覽工具。
'''
import time
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageOps

'''
Generate a fresh 32-bit seed only when -1 is requested.
只有設定 -1 時才建立新的 32-bit 隨機種子。
'''
import secrets

'''
Resolve one seed per run without overwriting the notebook setting.
每次生成決定一個種子，但不覆寫 Notebook 的原始設定。
'''
RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

'''
Require a supported seed before building the inference workflow.
在建立推論工作流程前，先檢查種子範圍。
'''
assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'

'''
Show the actual seed so a random run can be reproduced later.
顯示實際使用的種子，方便之後重現隨機生成結果。
'''
print(f'本次生成 seed：{RUN_SEED}')

'''
Verify that the requested mode matches the already loaded model.
確認生成模式與已載入模型一致，避免不小心使用舊權重。
'''
assert "QWEN21_STATE" in globals(), "請先執行模型初始化 Cell。"
assert QWEN21_STATE["mode"] == MODEL_MODE, "MODEL_MODE 已更改，請重新啟動 Kernel 後重新初始化。"

'''
Reject an invalid MMGP manager before starting an expensive generation.
GPU 記憶體管理器無效時，在耗費生成時間之前停止。
'''
assert QWEN21_STATE.get("manager") is not None, "MMGP 管理器未就緒，請重新執行模型初始化。"

'''
Ensure the requested budget was actually applied in Cell 3.
確認指定的 GPU 0 預算已在 Cell 3 正式套用。
'''
assert GPU0_MODEL_GIB == QWEN21_STATE["gpu0_model_gib"], "GPU0_MODEL_GIB 已修改，請先重新執行 Cell 3。"
pipeline = QWEN21_STATE.get("pipeline")
assert pipeline is not None, "模型尚未初始化，請執行 Cell 5。"

'''
Keep every generation instead of overwriting earlier output files.
自動編號保留每次生成的圖片，不覆蓋先前結果。
'''
def make_output_path(filename):
    base = Path("/kaggle/working") / filename
    if not base.suffix:
        base = base.with_suffix(".png")
    if not base.exists():
        return base
    for index in range(2, 1000):
        candidate = base.with_name(f"{base.stem}_{index:03d}{base.suffix}")
        if not candidate.exists():
            return candidate
    raise RuntimeError("輸出編號已達上限。")

'''
Validate the explicitly selected task before preparing references.
先檢查使用者指定的模式，再處理參考圖片。
'''
assert TASK in ("text", "edit", "multi"), f"不支援的 TASK：{TASK}"

'''
Require a positive output size compatible with the model.
確認輸出解析度為正數，且符合模型的尺寸要求。
'''
assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0

'''
Rebuild references for every generation to avoid stale images.
每次生成重新讀取目前選取的圖片，避免沿用上一輪參考圖。
'''
references = []

'''
Text mode never sends reference images, even when images remain selected.
文生圖模式不傳參考圖，即使清單仍保留已選圖片。
'''
if TASK != "text":

    '''
    Read the selection maintained by the independent upload cell.
    讀取獨立選圖 Cell 所維護的最新圖片清單。
    '''
    selected_paths = list(globals().get("REFERENCE_PATHS", []))

    '''
    Enforce exactly one reference for edit and two or three for multi.
    單圖編輯限定一張，多圖參考限定兩至三張。
    '''
    valid_count = len(selected_paths) == 1 if TASK == "edit" else 2 <= len(selected_paths) <= 10
    assert valid_count, f"TASK={TASK} 的參考圖張數不符，目前選取 {len(selected_paths)} 張。"

    '''
    Read and resize selected files without changing stored originals.
    讀取並縮放已選圖片，不更動原始上傳檔案。
    '''
    for number, filename in enumerate(selected_paths, start=1):

        '''
        Check the file before beginning expensive inference.
        在耗費 GPU 推論之前先確認參考圖存在。
        '''
        path = Path(filename)
        assert path.is_file(), f"找不到參考圖：{path}"

        '''
        Apply orientation and preserve transparent reference images.
        修正 EXIF 方向，並保留透明參考圖的 alpha 通道。
        '''
        with Image.open(path) as source:
            image = ImageOps.exif_transpose(source)
            has_alpha = "A" in image.getbands() or "transparency" in image.info
            image = image.convert("RGBA" if has_alpha else "RGB")

        '''
        Preserve aspect ratio using REFERENCE_SIZE as an area target.
        以 REFERENCE_SIZE 的平方作為面積目標，保留原始長寬比。
        '''
        scale = (REFERENCE_SIZE ** 2 / (image.width * image.height)) ** 0.5
        size = tuple(max(32, round(length * scale / 32) * 32) for length in image.size)
        references.append(image.resize(size, Image.Resampling.LANCZOS))
        print(f"<image{number}> {path.name} → {size}")
else:

    '''
    Make it explicit that any previous references will be ignored.
    明確顯示文生圖不會使用前次選取的參考圖。
    '''
    print("TASK=text：不使用參考圖片。")

'''
Validate the sampling controls before running inference.
推論前驗證採樣參數，避免無效數值。
'''
import math
from mmgp import offload
assert SAMPLER in ("auto", "turbo", "base"), "SAMPLER 必須是 auto、turbo 或 base。"
assert isinstance(STEPS, int) and not isinstance(STEPS, bool) and STEPS > 0, "STEPS 必須是正整數。"
assert isinstance(CFG, (int, float)) and math.isfinite(CFG) and CFG >= 1.0, "CFG 必須是至少 1.0 的有限數值。"

'''
Only the 6-step, CFG=1 combination uses the validated Viggle Turbo recipe.
只有六步且 CFG=1 時，才使用已驗證的 Viggle Turbo 配方。
'''
use_turbo = (SAMPLER == "turbo") or (SAMPLER == "auto" and CFG == 1.0 and STEPS == 6)
if use_turbo and (STEPS != 6 or CFG != 1.0):
    raise ValueError("Viggle Turbo v0.2.1 建議 STEPS=6、CFG=1.0；若要自訂，請改成 SAMPLER='base' 或 'auto'。")

'''
Activate the existing Turbo LoRA only for its matching sampler.
只有選擇對應的 Turbo 採樣器時，才啟用已載入的 LoRA。
'''
transformer = QWEN21_STATE["transformer"]
offload.activate_loras(transformer, ["0"] if use_turbo else [], [1.0] if use_turbo else [])

'''
Refresh any resident LoRA blocks after toggling without reloading the model.
切換 LoRA 後同步已常駐的區塊，不重新下載或載入主模型。
'''
manager = QWEN21_STATE["manager"]
if hasattr(manager, "refresh_loras"):
    manager.refresh_loras(transformer)

'''
Use the original Euler scheduler for the base model without the Turbo adapter.
基礎模型使用原生 Euler 採樣器，不套用 Turbo adapter。
'''
sample_solver = "viggle_v02" if use_turbo else "default"
print(f"Sampler: {'Viggle Turbo' if use_turbo else 'Base Euler'} | CFG={CFG} | Steps={STEPS}")
if CFG == 1.0 and NEGATIVE_PROMPT:
    print("CFG=1.0：Negative Prompt 不會影響生成。")

'''
Reset peak statistics so this run reports its own VRAM usage.
重置 GPU 0 峰值統計，只計算本次生成的 PyTorch 記憶體峰值。
'''
torch.cuda.reset_peak_memory_stats(GPU_ID)

'''
Execute the LoRA-accelerated sampler using the existing pipeline.
透過已快取的 Pipeline 和 Turbo LoRA 生成圖片。
'''
print(f"Generating: {MODEL_MODE} | {TASK} | {WIDTH}×{HEIGHT} | seed={RUN_SEED}", flush=True)
start = time.time()
with torch.inference_mode():
    result = pipeline.generate(
        input_prompt=PROMPT,
        n_prompt=NEGATIVE_PROMPT,
        seed=RUN_SEED,
        width=WIDTH,
        height=HEIGHT,
        sampling_steps=STEPS,
        sample_solver=sample_solver,
        guide_scale=CFG,
        batch_size=1,
        VAE_tile_size=(True, 256),
        loras_slists=None,
        input_ref_images=references or None,
        custom_settings={"qwen21_kv_cache": "Disabled", "rgba": "Disabled"},
    )

'''
Save the generated image without displaying it in the generation cell.
僅儲存生成圖片，不在生成 Cell 顯示。
'''
assert result is not None, "模型沒有回傳圖片。"
output = make_output_path(OUTPUT_NAME)
array = result[:, 0].permute(1, 2, 0).contiguous().cpu().numpy()
if array.dtype != np.uint8:
    array = np.clip(array, 0, 255).astype(np.uint8)
image = Image.fromarray(array)
image.save(output)

'''
Report the CUDA allocator peak and actual device memory occupancy.
顯示 CUDA 配置器峰值與 GPU 實際記憶體占用。
'''
free_bytes, total_bytes = torch.cuda.mem_get_info(GPU_ID)

'''
Show the requested budget alongside actual VRAM usage.
對照指定預算與目前實際顯示記憶體使用量。
'''
print(f"GPU 0 budget: {QWEN21_STATE['gpu0_model_gib']:.1f} GiB | actual VRAM: {(total_bytes - free_bytes) / 1024**3:.2f} GiB")

'''
Report the PyTorch GPU allocation peak for this generation.
顯示本次生成的 PyTorch 顯示記憶體配置峰值。
'''
print(f"GPU 0 PyTorch allocation peak: {torch.cuda.max_memory_allocated(GPU_ID) / 1024**3:.2f} GiB")

'''
Retain the model objects and report memory use for all visible GPUs.
保留模型物件並列出所有 GPU 的顯示記憶體用量。
'''
print(f"Saved: {output} | elapsed={time.time()-start:.1f}s")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GiB")
QWEN21_STATE["last_output"] = str(output)

## 9. 預覽與下載最新生成圖片


In [ ]:
'''
Read the latest output path and render the saved image.
讀取最新生成圖片路徑，直接顯示已儲存的圖片。
'''
from pathlib import Path
from IPython.display import Image, FileLink, display

'''
Ensure a successfully generated image exists before previewing.
確認已有成功生成的圖片後再預覽。
'''
assert "QWEN21_STATE" in globals() and QWEN21_STATE.get("last_output"), "尚未生成圖片，請先執行生成 Cell。"

'''
Resolve the most recent file recorded by the image pipeline.
取得模型狀態中記錄的最新輸出圖片。
'''
output = Path(QWEN21_STATE["last_output"])

'''
Reject missing outputs rather than displaying stale notebook content.
若檔案不存在則停止，避免顯示過期的 Notebook 結果。
'''
assert output.is_file(), f"找不到生成圖片：{output}"

'''
Preview the generated image directly beneath this cell.
在最後一格直接預覽生成的圖片。
'''
display(Image(filename=str(output)))

'''
Provide a clickable link for opening or saving the original PNG file.
提供可開啟或儲存原始 PNG 的下載連結。
'''
display(FileLink(str(output), result_html_prefix="下載原始圖片："))


## 10. ngrok 與遠端生成 API 啟動

In [ ]:
'''
Dependencies are available in the Python standard library.
只使用 Python 標準函式庫處理遠端服務與檔案。
'''
import zlib

import queue
import copy

'''
Use secure randomness when the API requests seed=-1.
API 指定 seed=-1 時，以安全的隨機來源產生實際種子。
'''
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

'''
Expose only the inference bridge, never the Jupyter or ComfyUI management interfaces.
ngrok 僅轉送推論橋接 API，不公開 Jupyter 或 ComfyUI 管理介面。
'''
REMOTE_PORT = 8765
REMOTE_SERVICE_ID = '06_Qwen_Image_2.1_圖片生成_Kaggle'
REMOTE_KIND = 'image_multi'
REMOTE_FIELDS = ['task', 'prompt', 'negative_prompt', 'sampler', 'reference_size', 'width', 'height', 'steps', 'seed', 'cfg']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda}Zms\x13\xc7\xb2\xfe\xee_1gS\xb7,QF\xd8&pO\xa8(U\x0e(\xc4\x15^|m\xc3\xb9u(\x97j%\x8d\xec=HZewe\xe3\xe3\xeb*\x9b\xc4\xd8\x8e1\xe6$\xc6\x04\xe3\x84\x90\x04\xe2\x00~\x01B0\xc6\xc0\x87\xfbOn\xbc+\xe9\x13\x7f\xe1<=\xbd\xbbZ\x19\xe7R\x15Y\xda\x99\x9d\xe9\xee\xe9~\xfa\xe9\x9e\xbc\'\xfe\x9a\x10\xd5\xc5;\xde\xccuwe\xa9:;\xfdvgf\xf7\xcd\xba\xb7\xf8B\xf4Xf\xb1\xec\x08of\xc9]\xf8\xa26\xf1\x05\xc6\xdd\xd7W\xdd\x85\x8d\xfa\xf4|\xed\xe7i\xf7\xce\xf3\xda\xdd\xabowf[\xde\x13\xdd\xc5\xb2i9\xc2\xaa\x94\x1c\xa3(E\xce\xb0\xcb\x05}T\xe8\xa5\x9c0\x8a\xfa\xa0\x14Y\xb34,-\xdb0K\xc21\xcd\x82\x9d\xc0K\xb5\x9dYw\xea\x1e\xef]\x9fx\xe2n\xdf\xffcb\x92\x85\xa8\xbd\x9a\xf5\x16n\xd7f\xa6\xeb?\\\xaf\xdd\x7f\xe5>\xbf\xe7N=\xffc\xe2r\x8b\xc1\x1b\xd1.-y\xc8\'\xca\xba3T02\xc2\x1f\xe8\xc1\xcf`R\xa9R,C\x06[\x94\xca\xe1{\xa6\x95\x1d\xe2\x17{\xbaO\x05/u\x93\x88m\xfc\xe7l\xd9n\x81l\'eIZ\xba#\x85.\xf2\x96\xb4\x87\xc4\xe1\xce\x83\x19\xc3\x11\xb6\x949a\x96\n\xa3bdH\x96\xc4\xc1\x0ea\xd8\xc2\x92\x9fW\xa4\xed\xc8\x1ci\xe5.<\xf0Vfk\xabk\xee\xfa2\x8d{\xb7.{\xb3\xf3\xee\xcb\xed\xea\xc39oi\xb3\xba\xfce\xb0V}y\xd5\xfb\xf5Nuu\xdd]\xbb\x1e\xd1\xcd\x96YK:J\x8a^i\x9b\x85a\x89\r%\xef\\\x96\x16\x19Y\x8c\x18\xce\x90Yq\x84\t\xa3\x8eX\x86c\x94\x06\x853$E\xc9td\xc64/b\xb6C\x0fI oc\xc1{t\x97\xcd\xec=\xde\x86X\xbb[\x13\xee\xc4\x1co\xfcv\xe7\xea\xee\xab+\xbb[\xf3\xb5\xfbW\xdc\x8d\x87\xe2L\xb0\x02\x04u\xaf\xddq\x7f\x99cUH\xc0\xdesg\xd2}\xa9\xd4\t\x91\x0c\x84LX8\xe2\x8c,\x98#\xb1\xce\x03\x07\x0ew\xc6\x85\x91\x17<%I\xca\xcb\x82-\x85Qrb\xf4,\xce*}^1,\xb2\xab])\x93\xbe\xd0J\xa9\x96\x91y\x13\xcf3\x15\xa3\x90\x0b\xd41Jyi\xc9RV\x8a\x11\xd3\xba\x98\xc76\xca\xc2+\xab\xbe9\xaf\xad\xd6\x1e.\xc19v_\xadx\xcf&\xab\xabs\xee\xec<\x14r\xa7f\xbc\x07?zw\xee\xb1\x8a\xd5\x8d/\xdd\x95yR@\xb7m\t\x0b\xb7\x8b\x0f\x93"T\xe6C\xa1Do\x13\xad\xea\xa7\xfbf\xaa\xfe\xc3\xcc\xeeK\x1c\xd6\x0ef\xd6\xa6\x9f\x8a\xf7;?x\xff\x83\xa3\xff\xd9\xf9\xc1\x11,\x8ep\x80E\xaa\x97\xb7\xa1\x1f\xd6l%\xa5\xfa\x86\xcc\x11%\xb0\x9eu*z\x81\x15\xb2M(I\xf6\x81\xb3\xd1\x99e\xf5\x12\x94\x84\xb3\x94-3W\xc9bF\x01\x0ef\x91F\xf5\xbb\x1b\xd5\x9f\xb7\xdd\x8d\xfb\xf5\xe5\xdb\xbb\xaf\xdeT\x17Wa\xfd\xf0|\xbc\xa5\x17\xbb\xaf\xdf\xec\xbe\x98C\xf4!\xf4\xaa\x0b\xaf}\xcfQGZ}v\xdd\xfb~\x85\xb4+[d\xe9|\xab\xb7\xf2(<p%\xca\xdb\x9d\xe5\xb1@\xdd\xf1Vu\x0c\xe7\xa5e\xe4G!\xb3\xee(\xc1C\x17\x16E3\'EQw\xb2C\xd2f\x9d\n\x96\xd4s\xa3\xa2`\xea9\x7f\xbc@BW\x7f\xdc\xae=\x98\xf7\xfdj\xf5\xae\xbb\xb3\x80xu\x9f?\xe1\xb0\xa6\'\xdf\xcf\xc1\xd3j\xd3\xbfA\x87\xfa\xe4\x1bwj\x1en\xe6n.\xb8o\xbe`%k3_y\x8f\xbe\x82J\x91\xb3\xd1\xfe\xebo\xa93\x9d\x1d\xe9\xbe\xfe\xae\xfe\x94\x06\x0f\x10\x83\x053\xa3\x17\xecX\xbcMh\xb5\x87s8\\F\x1e\xde\xc1\x9d\xf9\x0e>\xea^]\x12\xc7e\xa1\x80\x85\xb4`\xa5\xe8B\x174\x12[\x1b \xb7<}\xf6D\xeaT\x9a>\xb1`\xe3\x87\x80\xe8\xde\xed\xdf\x00|\x10\x17\xfb@,\x84\xab{\xe3\x8e;wC|&\xad\x92,\x08>\x00z\x1c\xec\xaa6Tn\xfd\x0f\x99u\x00x\x90xX/\x189q\xfa\xf4\xc9\x1e\xd8\xb1\x04L\xb1\x02\xe7\xb6\x1d\xddR\xb1\x8ay\xf2RY\x96l\x03\xd1=\xc8X\x03l$\xb3\x9e\xec9\'j\xab\xdfzW~\xaf?\\\xac\xae\xdf\xad^\xbf\xe2\xdeZ\xad~y\xd7\xbb1\x03,!\xf7^Y\xadM\xdc\xac=y\xe9\xdb\xfe\xd6\xe5\xfa\xd27\xe4\x1e\x80\x99\xc9\x15o\xed\xc7\x889\xa3FH\x0cJ\'\xa6\xf9"iqB.\xe0\x05"\xbe$\xc9\x12$o\xb8\x9f\xb7\xf2\xc0\xdd|\\}\xfeu\x931\xf6\xb3{`\x81T\xc9\xae@\xc5fg\xcaTr\xd8T\x8c\x00\x8494\x00\x9cz\xb9\\00\x86\xa3\xa5#\x13\x87\x1b\xce\xe4]\x9d\x06\xd4\x10D\x92\x15\xda\x05\xd0\xbf\xba~\x13\x07\x03\x95\xfd\xc9\xc2[\xfb\t\xae\xe6\xde\xbb\t\x07\x8a(\x8a\x17\xda\xd3|\x9a\'\xbb?\xa6sn>\xff\xc1r\xa5=\xad|7=hd\xb4\x01h\xbc\xe7\x15l\xc3\x89\x8fU\x86\x9fE\xb5\xf6\xb7W\xda\x96\x8d\xb2,\x18\xc0\xe4\xe4>\xf6\r\x06\xb5x Z8\xbd\xd9\xe0\xbe\x1d7\x97\xc9\xd6\x815\xfd\xbd#[\x1e\t\x0c\xfc\x99\x94e!\x01\xf8\xa3\x11\x8f\x81\x15ah\x1d9)\xdf\x94\x0c\xa4n\xc1\xc8\x96@\x92(#O\xe4\x8d\x82\xe4\x9c;\xfd\x00\xfe\\}\xbeZ\xbbug\xf7\xcdw\xd5\x1b\xb7\xa2\xe9\x810\xdf\'\x01W9%\xd4\xbeQ\x017;\xdf\x80\x99\x9c\xcc\xc3\xaf/\xca4\xaf\x9d\xa6\x0c\x1c\xa3\rJzQ\xc6\x8f\xb5\x08\xfc\xcb\xe86Y\x87\xb2qL;tQ\x1f\x1c,\xc8C\x84\xe1\x90\r\x9ewH\x04\xf3\xd5l\xe4\x0c\xb2\x0b\xbd\x94\xb0+\xf9\xbcq\x89W\x89\xac\xa4\xc6(\xeb\xa5yBLK\x94i\xa9w\xde\x97\x97\x0c\xdb\x01X4V@\x9e\xaaX%5\xaa\x9e!\x12a\xb5\x9c\xbcD\x1e\x08|\x1e\x941\x80\x7fG{{{\xe4%\x80u\xce\xc8\x11\x01\x88\xeeM\x12\xc7\xf2\xda\x18K\xea\xc8\xe2xzL-u\xac\xfdpn|,\xa2\xc0\xb8/ZD\xbcp\xc9}d\x8c\xc8\x19NS\x83\x96n@\xff^\xe6U)\xcb2\xad\x98V\xdb\xd9r\xa7\xb7\xf9\x0c\xe1\xb4\xf5\xc9\xc5\xdd\xad\xaf\xea\xb7\x88Ch\x0c\xee\x04@$;\x05#p\xa6`d\r\x07\x91g\xcb\x02@\n\xa1\xe7\xe8\xf6\xc5\x00\x93\xcaHH\xbaE>cI?\xdb*O\t\x93\xa8\x8f\xd5\x13Sat2\xda\x13\x12]\x99\xaf\xdd\xbaA\x88\x11\x10D\xf8N$(\xfb\xbb\xfa>#+\xc74G^r48\xbd\xcc\x19\xeao\xb1Rp\x0c\r\x90\x9e\xd7\xe0h\xde\xe2\x86\xb7\xf0\x1b\xc5=\xbdA\xe9\x8a\xfe\x8ek\xcd\x84\xa1l\xdapn\xa2C\xec\xd4\xb6\xf1O"\x96E8\xa0\x91)HE\x8a\x94\xca{R\x14\xdb\xab\xf6\xcbO\xde\xf7\xd7A2\x91\xba\x81!\xde\x8d-\xe5\xe4\x8b\xd5G\xf7\xdd\xeb3\x1c\x8c\xe4\xff\x9b\xc8\xc3[\xb5\xfb\x93\xde\xe3\xcb\x11M\xfe\xd6}\xa2\xffS\xf1\x110\x898\xed\xa7\xa9\xee\x93\x9f\xf6\x87?y\xf0?\xc0\xeb\x08w\x9a\xa6\x84\x0fY\x15\xc5n"\x86V\xce\xf8NL;`\x0e\xc3&r\x08\xf2\x05\xf4R\x0c\xda\xdeK\xe5\x18\x9cj\xeb\x13\xee\xc2R\xf5\xf6:B\xb4>\xb9E\xdf#1\xccY\xd7{B\x07\x08\x1f\xa1l\xbc\xf3 <+R/"JR\\\x18 )\xfbqT\xcc\x02J$\x19\xbc\xa6\x94\xb3\x1b2\xfb\xe2\xb4\x91\xd8%f\xc1\xfc\x04S\x8a:\x0e;\xf02%\xf0\xd24\xa4\xc5^\xec2\x94\xff/?\r\x05 \x0f\x9a\x7f\n\xff\xf2\xb6\xa6\xdc\xa5\xf5\xdd\x97\xf3\x0cI\xca\xa9\xb7\x1a\xde\x84\x00R\x9e\xf4\x97\xa4`G\xe2\xc0!\x83\x02\xf9\xe8\xc4yO\xb2\x1d\x89\xe0\xe0?J>\xa3>\x89\xccI\xa4\xda\x9c,9\xa2R&\x16#\xb2\x00\xd6\x84\xbf\x86o\xc1\xf9U\x10J\xde\x95q\xd7\x9b\x9d\xa8\xfe\xbeV[{N\x0e\xbf2A\x89@\t\xc4\xb2\x92X\xf4~\xa0\xacBA\xb2a\x01\xa1\x1d\x0b)\x0b\xa7\x84\xde\xd4\'\xa9\xde\xd4\x99\xe3\xa9tOW\xff\xa7}\xf0\xff\x0b\x03qD*\x0b\x90*\xc1\t\xb2\x14\xa8H\x91\x88R"\xfb\rk+\x07A\xd0(\xa7rFL\x81\xdf\xce\x90%yD\x85Q\xa0\t\xc4"\x11\x81\x0b;\x1b\x80\x03\x9f\xe8\xef<#3\xff\xbcL\xd5\x9b2<\x0f\xb9S\xbf\x82\xda\xeen\xcdbB\xa0\x8c".\xe9\xac\t\xb4!Md)\xd6\xac^\x9c<\xb9C\x04\xc7\x81\x1f\x1c\xcfL\xf3;\x89U\xef\xf7\x12\x1ew\xb4\xab\r\xfc`\x8a\xecC\x00@\x8b%9\xe2U\xd5\x11\x16\x9b;\xcf\x10\xa5\xf0\x19\x84(t\x88:\xb9\x18\xdbg\xa3q\xc1\xbah-Q\xf7 \xbb\xa1\x82#\xac\x08\xf1O\xa5\xc3\xb0\x88\xca\x0e!\t\x10\x00\xda(\x0f\xa9\xbc\xb3\x0c\xfc\xd4\xa9H\x8d\xba\xc8\xee\xd6\xfd\xea\x16\xf8\xc1\xeb\xa8{r\xa2\x04oDV\xe5r\tq\x06\x1f\xf7\x1e,zwg\x02\xc3\xd2Q\xa1\x1a\xcdH\xab-\xcc|\x84\x8b\x12\x0fU\x99\xb9G\x956f\x8a\xc9\x8eH\x8axO\x1c\x1f\x92\xd9\x8b\xca\xa5i\x8d\xb0`\x92\x10\xb6\xa4r~\xc8(\xc3\xba)\x11y=\xa4\x8d\x8a\\q\xed\xe4\xb3\xc6\xa9\x19F\xca\x86\xe9\xd7\xbe\xc5\xf4@|\xfaGr\x05\x19=L\xf6\xe1h\xc0t0\x9a0\xec4M\x88)p\xf7f_S\xd0\xcflF\x82~y\x8c\xe6\x8d\xfb\xc7\xc4\xc2u\x81\x14\x8e\x92\xe5\x11\xa5\x0c\x82tn\xc8MX\x17\n9\xc8\xd36\xd2\x14\xc5\xf0^$\x8a\xeaH\x04n\xed\'\x91\xfa\xef\xeeO\x80=/\xdc\xeb\xffR\x07t\x9fq\xa5>\xb1\xe0}{\xad\x91\xab\x90o\xf4ByH\x17\xf5\x89\xe5\xfa\xe47QuU.Q\r\x81\x84\t\xab\xc6H\xe28\xf5\x12l\xb3\x82`m\xce\xdc\xdc\xe2H\x86\r\x04J\xf0\xf94\xcbl\xda8\\\xf5N\xbc\xe9\x9d!\xddN\xf3\xe6\x88\xa2.U\xe8\xa8e\x0812\xd0\x1d\xe0A\xa1\xae54\xcf\x8eFf\xe1\x84\xcd}e\xe0an\xb7\x10\xf4\x9c\xfc\x98\x16\xcfG\xf6S\xc1J\x03Z<z\x02=\x81\xadu\xbbL\xa5\x8c\xcaE\xa2b\x93g5\x00\xac\xaf\xfb\xef)2\x03\n\x18\xc8\xa4\x83IX\x10\xb8\xe9\x08^\xde\xdb;\x9f\x82\xfa\xc5S*c_\xad \xfb\xd6\xbf\xfb\xb1\xfa\xeb<\xc2\xd9[\xbdE\xc7\xc3\x98\xaf\x82\xa7~\xe3\xb9\xbb\xf1\xc8\xdbX\x8c\x9e\x85\x9d\xa5,\x98\x14\xb1=\xcb\x1e8\x00\xcc9$b\xac\xf2\x88\x91\xc3\x91\x1d\xf0\r0$\x8d\xc1!\'\x1e\xa7I\xed\x89#\x8d\xb5\x08\x07\x92\xc2A"\x90\xb1\xa2~)F\x85\xbf\x05$\xca\xc5\xe0\xd3\x83j\x05\xde\xef\x90\xa0v\xc6\x01\xf5I\xf1\xeb\x0f\x87\'@+\xc5#$3\xe4N\xa8o\x90m|\xa9\x18yb\xf4\xe17\x98\x12\xbd\xd2\xd6\x8b\xe0d\xa5\xc1\xc4\xa9\xae3\xc7\xff~\xb6/\xdeX\xc7\xaf\xe1\xb5\x0f\xd5\xebc\x0c\x1a\xe3\x1f\t\x152\t\n\xbaq\xf1\x7fW\xbe\x16c\xb4"\x91L:\xcd \x1d\x9e\x06+\x17H\x15\x01\xe5\xe3\xf2^/\x8dR \r\x1bf\xc5\x8eR\x8f\x11\x03Y.\x83\x17\x06K\x04{\x01\xd8!B\x00\x06\xdc\x91\x08\xd37a\xdc\x8a_\xb5\x030@B\x1ad#J\'\x1a*0\xaaS\xb6F\xc0\xe3u\xff\xddf\x9e\xb8\x0f]\rlC\x1dC\xc72\x0bv\x80sV\x85Q.\x82m\xef\xf9(Fi\xe1\xd7\x9b\xb5\xb5Y\xef\xda]o\xf5\'l\xc2\xec\x8e\xe9\x8f_:\xdf\xd8r\'v"m\xb6"u\nU\'\xb0X\x1c,\x07\xad@3\x9f\'\x86\x10\xd0\xbe\xbe\xae\xd3=\xa7R\xbd\xcca\xf5\x8ac\x12w\x05M\xcf\xa8/\xc4\xf7\x89\xc3j\xc14\xee\x1dy\xdfn\x08\x9a\xfb\xc7\xc4\xa4\x9aJ\x1dTUDD\xbb\x13\x86M\x95\x9b\x0e=b}\xfd\xa9\x9e\xbe6j\x93\xc5\x15\xeaQ\xb1\xf0\xeep\xc64\x0b<\xae\x1e\x10\xf9\xa4\x9d\xd5\xf7p_El\x7f\x83\xae\x7f\xb2\xd5\xf1ON\xb6!`T\xde\x85\x9e\x14 \xb4b\x91q;o\x94\x0cGM\xe2\xc7\xf8">B\xf2N\xd0N\xf4#\xdc\x07\xd4\xc1\xdd\xfc\x17\x8d\x08E\x8ffA*B\x0b+\xda~\x96z\xa4t\xa0G\x0f\xa2P*\xb7\xd1b\xc9\x0e\xa2\xeb\x19\xe4V\x85\xf0\x15\xdb\xef0\r\xfb\x0e\x90\x13\xe7\r\xaa\x14E\xbf2\x9b%\xb3\xa8\x9c\x1b=Uwj\xcd[\xbb\x07\xca\xee\xaf\xc5\xdd\x10o6\xf0-\xa4e\xe5\x05\x84\xebM\x0b\xd5\xc1\x81\x97^\xd0\xd9c\xcb4\x9f\t\xe0$85\xe21|\xa6\nr\x9b\x9e\xab3\x0f\x8d\x91T\xc6\x88\x1c\x02\x1e\x1c\x8d\x13?m,L\x83|f\xc4W\x8f\xd2\x8a\xf4\xea_\xd4\xab~>\xe7\xaa\x0e~_\tj\xba&y\x87\xdb\x13\x9d\x89\x0e\xe1\xbe\xdc\xae\xadm\xf2N\xc9\xa3\xf0&\xa5v\xa2\xfd\xed\xce\xed\xda\xdc=\x94((\xe6k\xab\x97\xb9e\xe0-\xbe\xa0v\x9f/{\xb2\x95\xfc\xadUy^+\xe9\xd0\x1aF[\x17x\xf2p\xa38\x04]\xa5\xa8\xe2\x8dO\x99\xbd]\xdc\xddVE\xb2cs#Pq#\x8aK\xeeV\xf2i\x00\x00\xbco\xa6\xdd\xcdyoZY\x9cW\xf0#\xf0\xd6jx8\xee\x8d;|8~\xbb\x1fSi\x1b:\r\x95\xdb\xb0\x138\x90\xd8\xdb\xaf\x89\x8ci\x03-~X&t_\xf8t\xc1\xb4t;\x16\x99\x04N\xad\xb5k\x03\xa2\xe9,T\xaa\xbb0\x801\x98m\xff1\xbf)\xcdM~\x82JBlU+(cd\nf\xf6"\xd2]\xde\x81\x8c\x8e\x89C"c\x04\xcc\xd1\x92$U\xd0\xad\x0e\xcbNwf\xda[\xb8\xcd\x0b\xd0\x8d\xc9\xf5\xab\xf0[\xeabmm\xd5\x7fQM\x96\xf9\t\xf7\xeeW\xcc\x1d\xb9\x8e\xdb\xdd\x9a\x83}\xa8\xb7\xac\xac\xb4\xbb\xf5\x92+R2S\xd0D\xdck\xa2\xa0\x937\xd0\xc2\xf9]w\x1c+\xe6?D\xc0Z\xac\x14\x9bJ\xf3=\xcf\x1fN4\rF\xed\xa8\xecq\xcef\xf7\x08\x98\xb0HU\nT\tf\x87dN}\xcb\xab*DrwF\xe9\x1d\xda\x84\x1e\xb3/\xe89\xbd\xec7\xb8\xdd;\xdb\xd5\x9f\xaf\xf9=a\x8e\xd6k\xd4\xbf\xf6\x17\x0e\xbd\x86M\xc2\xcd\xbd\xe6U\xc8\x10\xec\x83iu\'B\xe6\xd0\x86U\xd4\xa4\x87\xdb;\xb5}\x0eW\xcb\xc9\xbc\x8e\x02I\x0b:\xe3Z\x1f;\xf111\xd6\x1a\r\xb8\xd6}^n\xfd\x98TS\xe2\xb5\x8e\x8b\xffQ\x903\x86\x0f\xfa\xde\x07@\xb3\x93c*0)\x07\xe3\xed=\xe8p&u\xb2\xab\xbf\xfb<\n\xbe\xde\xb3\xa7{\xfa\x8fE\x13c\x18\xc5\xcbg\xe4\xa0\xae\x1a\x1a\xfeM\x1bgY\xf7\xd5\xe3\xfa\x9d7\\\xe4\x87Q\x0b\xd2 \xc1\xaf\xa5~\x91J\x03\x87\xa26Kd\x14\xe6\xa6\x0b\xa8J\x89\xee\x15\x90\xc0l\x15\xb5\xe6HI\x9c\xef\xed:\r\xa5\x88q\xd0\r\xc3\xf4|\xf5\xd5\xba\xdfVu\x9fn\x02\xa9\xab\xcf\x1e\xd7Vg\xa88\\\x00\x8c\xccT\xd7oF/\x0e(N{F\xfb\xe9\xbe\xac\xd1\x8e\xe6\x17U\xe8\xd2@"[\xc9\xe9Dq\xa4\x93&\xd1\xd2EY4\xad\xd14Ih\xc7\xb0W\xba\x9b/}R\x97d\xb6\xe2C\x0eE\xc5A=\x8b\x12\\U>\xb9\x00Y|\xaa\xd9\x84JA\xe7Ti0\xb1P\x9f\xbcFQ\xf4\xe6!3\x0f\xd1\x134V\xdd\xaf\xafF\x11,z\x9d\x19\xb9\x18\xd1\xfcK=\xac\x0c\x0fht\xfe\xe9H\xfd\xf2\x13_T3g\xfc\x7fo\x8eq\x07\x87\x9e\xd1\x1dJ\xb2q\x83\xa2Q\xf2\xac\xd8C\xc9~\xab\x02\n\xa8J5\xe2\x94FQ&\xe8#\x16o\xe1V\x94\xb2Q\xc8WT\xf79h\xfa\xc1h\xf0L\xbc\x15\xaa\xe8\xf7\x7fd\xac\xd17,\xa9\xf6\xaa\xf2\x8d$;R[8X\n\x06\xf6\xb8Zc\x86\x92:\x10\xba-R\xd9\x80\'\'\x95\x96\x8d\x87L\x97\x93\xacrd\t\x9f\x8c\xa5m\xe5\xf1\xccD\x9aG\x83pL6\xfdjL\x1a\xac\x00M\xd3\x8aR\'\x89}D\x1a\xbb\xc8,i\xa2\xb1\xc9\x8e\xc6\xd3\xf3]\xa9\xb4c\xd0:4\x10#\x0b\xb7\x89\xce#G\xe3\x8d)\n\xb2\xd265Z\xec\xa4\xea\xa5\xef\xb1\x18\xb0-\xcd\xa5a2Bw\x81X\xcd\x93\xb3\x15\xd4\xfb\xc5\xb4\x7fS\x8ax\xd6>\x1f\x91%\xc0\xeb\xc5\xe1tV\x07\xd4i\xc7\x84v\xc2\xb0\xf5LA\xe6\x88\xeeY\x83\x19\xbd\xe9\xd98/\xa6\\\xbcO\x1ff\xff\x0e\xce1\xb8\xf9\x0ep\xd1\xbf\x13W\x14\xd6!B\x19\x99Ll\x88{R\x80\xca/\xa6\xdc/\x9f\xa0\x00o\xbe\x94W\xb8\xb8\xb2\xea\xdf\xea\xa9\x0e\x15s\xf4H\x93\xd2w\xaa\xfd\xae\x19\xbc\'_\x13\x83\xba\xfd=\xf5\xdfB\xfa\xdd\xe2wR\x93\xef6\xf4\xcf\x9e\xeb\xef9\xd7\x9f>\xd3u:\x15o\xd1-K\x1f\xc5,\xde\xe0\xc2\xb16\xd1>\x90(K\xab\x88\xa0\x8eu\xe0\x80\xf0 NE\xa7c\x0cVPf\xc4\xf0\xa3\\\xc1\xa7\xba\x82\x8f)\x84Tk$r\xcehY\x12\x19*\x95\x13\x15D\xe5_9\x1c\x82\r\xf04[0\xca1\xf5\xbb\x8d\x18n\xe7\x91#\xf1\x84n\xd3k\xb1\xe0\x1d\xac\x17\xad\xb8\x13\xc4\xe2\xd5\x1b\xfc\x9e?\x9c\xb0q$1\xd6\xc9\xc7P\xfe\x7f\x00`\xf7\xe3\xe7Nt\xa1\xf6Gr\xd7\x11\xa5\x8c\xab\x84\xdb\xfe\xfdm\x0e\xd5R\x16\x89M\xc1\x990\xb3\xd9J\x19\xf4y\xb4qU\xcb\x0b\x80W\x02T\x91\xb6\x18\x17k3\xd3\n`\xf9\x1e\xb7\x01\x9a\xf3?\xf8WUH\xb92\x9d\x19u\xa8{\xea\x98\x8e^\xe0\x1f\x84\x1d\r8\xc5\xa6i\x94\xd4i\xaa\xf0\xa3\x18\x1a\xde1\xbfs\xbb\xa6\x17L\xf8/\xc2,\x10\xbf\x19\xfaA\xd1\xaaS\xbfp\xf3\x9e/\xd4 (7\xd3XRV)\x94\x97\xf3s}z\xa1\t:U\xe2\xe0\r\x01\x9eMD\xa4\xb5\xf9n\xadu\xe0X\xa2#?.N\x1a\x1f\x03;#"\xe1\xbdXT\xed\x83\xa2a\x10\xba\x13\xeah\xef|\xff\xc0\x81\xc3\xc7\x12\x9d\xfc\xb6\xb6\xf7\xd4\x82|D\xc2\xf8\x87G\x91\xa3N\x8f\x19\t\\\xbf\xf96\xd5\xafc\xff$\xb1\xed\xd1\xdc?\xcf0\xc95+\x1f\xbc\xb4ggh\x15==\xfdR\x90\x05\xfdy2\x17\x9c\xe2\xff\xa3"u\xac\x1b\x1cR\x98\x19\xbaM\xb6\xfd\xde%\x97\xab\xec\x8b`)JU,.\x86Q\xd8\xd1\xbd\x07\x96W\xb7\x03\xfe}\x1e\xdfd\xcc\xfe\xba\xfb\xf2\xf7\xdd\xad\xfb\xee\xccMwz\xdb\x9b\x9d@\xf8+\xc3A\xff=j\xbf{\xda\x04f9(\xc6\xc1C)P\x16\xf4\xb2MY0\x92\xe5\x0e\xaa\xcc\xa7\x0e\x1b\xfc\xb2E\x95\x0c\x8d;\xb5\x88Q8\x9c\xb8\xe1\x1b\x8b\xc7\xa3l\x88\x8d;f\x8c\x8b\xd0\\{\x0c\xba\xd7\x98\xc6\x9f\xd8\xb1\x99\x1b\x17\x00\x19>\x9e\xd1\xad?H\x93\x15@\xc1\xbf\x01v\x93x\xe6').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['text', 'edit', 'multi']
REMOTE_MAX_IMAGES = 10

'''
Keep job directories isolated between different notebook services.
各 Notebook 使用獨立工作資料夾，避免不同服務互相干擾。
'''
REMOTE_ROOT = Path('/kaggle/working/remote_api_jobs') / REMOTE_SERVICE_ID
REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('REMOTE_ENGINE_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=32))
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(2))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = 24 * 3600
REMOTE_RESULT_BYTES = 8 * 1024**3
REMOTE_MIN_FREE_BYTES = 2 * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = 2 * 3600

'''
Report only checks that can be verified without loading another model.
只檢查不會額外載入模型的服務就緒條件。
'''
def remote_backend_ready():
    if REMOTE_KIND == 'image_multi':
        return bool(globals().get('QWEN21_STATE', {}).get('pipeline'))
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        return globals().get('pipe') is not None
    if REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        gpu = globals().get('GPU_ID', 0)
        process = globals().get('SERVER_PROCS', {}).get(gpu)
        return bool(process and process.poll() is None)
    base = globals().get('BASE')
    if not base and REMOTE_IS_LTX:
        base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
    if not isinstance(base, str):
        return False
    try:
        with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=3) as response:
            return response.status == 200
    except (OSError, ValueError):
        return False

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'backend_ready': remote_backend_ready(), 'stalled': remote_stalled(),
                                 'running': running, 'waiting': REMOTE_QUEUE.qsize(),
                                 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return

        '''
        Reuse the same job ID for retry attempts after an uncertain response.
        網路回應不明時，重試會沿用同一個工作 ID，避免重複生成。
        '''
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        if not remote_backend_ready():
            self.send_json(503, {'error': 'Backend is not ready; execute model initialization cells first'}, 20)
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):

                '''
                Ignore missing seeds and validate the supported 32-bit range.
                未提供的種子沿用 Notebook 設定，並檢查 32-bit 範圍。
                '''
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')

                    '''
                    Translate -1 into an explicit random value before inference.
                    推論前將 -1 轉成明確的隨機數值。
                    '''
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)

            '''
            Save supplied seeds for reproducibility in the job status API.
            在工作狀態 API 中記錄請求所使用的實際種子。
            '''
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        '''
                        Point R2V to this job's uploaded reference images only.
                        讓 R2V 僅讀取這筆工作的參考圖片資料夾。
                        '''
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:

                    '''
                    LTX-2.5 uses a shared ACTIVE_JOB configuration.
                    LTX-2.5 使用共用 ACTIVE_JOB 參數。
                    '''
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'

            '''
            Clear old output metadata so a failed generation cannot reuse another job's file.
            清除前次輸出狀態，避免本次失敗時誤取其他工作的結果。
            '''
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)

            '''
            Execute a fixed local generation cell, never user-supplied code.
            僅執行 Notebook 內固定的生成程式，不會執行請求傳來的原始碼。
            '''
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):

                '''
                Include YuE2 replay metadata and score with the single MP3.
                將 YuE2 的重播參數與 ABC 樂譜一起提供下載。
                '''
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

'''
Keep one worker and one maintenance thread even after rerunning this cell.
重跑 Cell 仍僅保留一個 worker 和一個清理 thread。
'''
if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()
if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

'''
Close only the previous ngrok tunnel; preserve the running API and its jobs.
只關閉舊的 ngrok 通道，保留正在運作的 API 與所有生成工作。
'''
old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)
NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(Path('/kaggle/working/ngrok'))
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

'''
Always request the auth token again when this cell is rerun.
每次重跑均重新要求輸入 ngrok Authtoken，不顯示內容。
'''
ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')
result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')
del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

'''
Reuse the same API server when this cell is executed again.
重跑本格時沿用同一個 API Server，不中斷生成工作或清空佇列。
'''
server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()
if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:

    '''
    Update the handler on rerun without restarting an active GPU job.
    重跑後更新 API handler，但不重新啟動正在執行的生成工作。
    '''
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')
with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)
if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])
print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')


## 11. Web API 使用說明

In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request

'''
Request the API schema directly from the local API server.
直接從本機 API Server 查詢支援的參數與模式。
'''
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)
print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本地 requests 範例：06_Qwen_Image_2.1_圖片生成_Kaggle_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')